# Lab 4 - Classification

**Name:** Kris Su
**Date:** October, 2026
**Class:** Remote Sensing Fall 2026

This file contains the answers to the worksheet for `lab4 - classification`.


In [ ]:
from PIL import Image
import rasterio
import numpy as np

a=np.array(Image.open("/private/tmp/hw4-432.tif")); 
lo,hi=np.percentile(a,[2,98],axis=(0,1)); 
print(a.shape,lo,hi); 
b=np.uint8(np.clip((a-lo)/(hi-lo)*255,0,255)); 
Image.fromarray(b).save("/private/tmp/hw4-432.png"); 
Image.fromarray(b[:850,:850]).resize((1275,1275)).save("/private/tmp/hw4-nw.png")

(1626, 1656, 3) [ 8. 13. 17.] [92. 46. 40.]


**Landsat TM Spectral Bands:**

1. Band 1 (0.45 – 0.52 µm): Visible Blue [not shown in composite]

2. Band 2 (0.52 – 0.60 µm): Visible Green [displayed as Blue]

3. Band 3 (0.63 – 0.69 µm): Visible Red [displayed as Green]

4. Band 4 (0.76 – 0.90 µm): Near-Infrared (NIR) [displayed as Red]

<!-- Used for water penetration, bathymetric mapping, and separating soil from vegetation.  -->

<!--  Measures green reflectance in healthy vegetation and assesses plant vigor.  -->

<!-- Senses chlorophyll absorption for plant species and health differentiation. -->

<!-- Highlights biomass content and delineates water bodies. -->


## Supervised classification


### Question 1

```text
C   DISTANCE
L
A
S  0       2.7     5.5     8.2    11.0    13.7    16.4    19.2    21.9    24.7
S  |-------|-------|-------|-------|-------|-------|-------|-------|-------|

  8 -------------|
                 |-|
  7 -------------| |---------------|
                   |               |
  6 ---------------|               |
                                   |---------------------------------------|
  5 ---------------------|         |                                       |
                         |---|     |                                       |
  4 ---------------------|   |-----|                                       |
                             |                                             |-
  3 -------------------------|                                             |
                                                                           |
  2 -------------------------------------------|                           |
                                               |---------------------------|
  1 -------------------------------------------|

   |-------|-------|-------|-------|-------|-------|-------|-------|-------|
   0       2.7     5.5     8.2    11.0    13.7    16.4    19.2    21.9    24.7

```

From dendrogram, the features that are most similar to each other are `dark urban` and `bright industrial` as they merge at the smallest distance. This makes sense, while preparing the training samples, I notice that bright industrial patches were generally smaller and surrounded by larger areas of dark urban cover. This suggests that they present different surfaces within the same built-up areas, which is consistent with their spectral similarity in the dendrogram.


### Question 2:

<!-- Are there any obvious errors? If there are systematic errors, can you explain them based on the reflectance properties of the classes in which errors occurred? -->

In the results of my supervised classification, I found two obvious errors.

1. The estuary of _the Back River_ and _parts of both water reservoirs_ are classified as `bright industrial` land patch. I examine the spectral signature of the training class reconstructed from the `.gsg` file for an explanation to this error.

   Water absorbs strongly in the near- and shortwave-infrared, and the misclassified reservoir and waterway pixels retain low values in these bands. However, water's spectral response varies with its optical conditions, including scattering by suspended particles. The water training samples appear to represent too narrow a range of conditions, allowing some water pixels to receive a higher likelihood under the more variable industrial class.

   The signature statistics show that the water training samples had relatively little spectral variation, whereas the `bright industrial` samples were much more variable. Those water pixels therefore fell outside the conditions represented by the water samples and received higher likelihood under the broader industrial signature. This suggests that the water training regions did not capture the full diversity of water in the scene; or that the industrial training sets were too board, that encompasses signatures

2. Comparing the supervised classification with the original image, I noticed that the `pink-field` class appears to extend into `maroon` and `dark urban` areas, particularly along Park Heights Avenue. Some patches that appear to be `maroon` were assigned to `pink fields` rather than `maroon vegetation`. This may reflect spectral similarities between trees and herbaceous vegetation. At the image’s 30-meter resolution, residential pixels can also contain mixtures of trees, lawns, roofs, and roads, which may make them difficult to assign to a single land-cover class.

<!-- 2. Comparing the original image, `Baltimore_tm_subset.img`, the supervised classifier also included more area as pink field than there are. The mistaken `pink field` are mostly `dark urban` and `maroon vegetation`.

      This could be due to that, between the `maroon field` and the `pink field` are likely similarly to each other in the VNIR range, given that both of them are vegetation of different type.

      Regarding, pinkfiled intrution into the the pink field intrusion into the `dark urban` area happens more as a secondary effect of `maroon field` being classified as `pink field`. This issue is most pronouncing along `Park Heights Avenue`. Tree canopy are classified as lawn  -->


### Question 3:

<!-- What are “Maroon Vegetation,” “Pink Fields,” “Dark Green Vegetation,” and “Bright Green Fields”? -->
<!-- Symbology to assign Layer 4 to the red display channel, Layer 3 to green, and Layer 2 to blue: a 432-RGB combination. This Landsat TM image was acquired on October 19, 2009. -->
<!--
Layer 2 [visible green] - blue
Layer 3 [visible red] - green
Layer 4 [NIR ] are assigned to red

near-infrared are displayed as red, visible red as green, and visible green as blue.

-->

The class name for the following land patches describe colours in our 4-3-2 false colour composite,

1. `Maroon vegetation` are likely forested areas. These patches have irregular boundaries and a textured appearance consistent with tree canopy. Leaves strongly scatter near-infrared radiation while chlorophyll absorbs visible blue, producing a dark red appearance in the composite.

2. `Pink fields` are vegetated open land, including grass, pasture, lawns, and possibly grown crops that was still in the field. Their pink color is consistent with vegetation producing a strong NIR signal. But their relatively smooth appearance and defined field boundaries distinguish them from the previous class.

3. `Dark green patches` are likely agricultural fields with little green vegetation, or even exposed soil or senescent crop cover. Their field-shaped boundaries support this interpretation. Compared with the pink fields, their spectral response is less dominated by NIR, so the red displayed there is less prominent.
<!-- reflecting less NIR, dead trees reflects less visible-red, trees w/o leaves -->

4. `Bright green patches` are sparse vegetated field, exposed soil, or dry residue. They are brighter than the dark green fields and consistent with the greater exposure of soil.
<!-- agricultural fields, plausibly harvested fields with exposed soil or dry residue.
doesn't reflect, brighter reflects more visible-red -->


### Question 4:

<!-- This is an October image. Have the trees lost their leaves yet? How does this impact your ability to classify the image? Are there some land cover details that might be easier to detect in a different season? [You might want to add P15R3320100123.img, a winter image, to your Map Document when considering these questions] -->

![October and January Photo](figs/winter.png)

**The trees have not lost their leaves in the October image,** as indicated by the extensive red and maroon colour covered. Leaves strongly scatter near-infrared radiation, which is displayed as red. This helps distinguish wooded vegetation from other patches like high density residential land and water.

In the January image, many of the same of wooded areas appear brown or muted rather than maroon, consistent with deciduous leaf loss. With fewer leaves, branches, and leaf litter contribute more to the recorded signal. This changes the forest's spectral signature and can make some vegetation classes harder to separate.

Tree canopy is an example of such thing that is easier to identifying across seasons Winter imagery can help distinguish evergreen from deciduous forest because evergreen trees retain foilage. Growing-season imagery can help distinguish actively growing corps and grass from bare or harvested fields. Comparing seasons therefore provides additional information for classification, although some dormant fields may become more similar to exposed soil in winter which makes it harder to tell them apart through remote sensing.


### Question 5: Report the area of each class (including units!).

<!-- Report the area of each class (including units!). This can be calculated using Toolboxes → Spatial Analyst Tools → Zonal → Zonal Statistics as Table, specifying Classvalue as your Zone field and your classification as the input raster and input value raster. The output table will appear under Contents, and you can right click to open it. -->

See table below for the area of each class in the supervised classification,

| Class value       |   COUNT |  AREA (m^2) |
| :---------------- | ------: | ----------: |
| Dark water        |   15478 |  13,930,200 |
| Bright water      |  119235 | 107,311,500 |
| Maroon vegetation |  743252 | 668,926,800 |
| Pink field        | 1028605 | 925,744,500 |
| Dark green        |  112671 | 101,403,900 |
| Bright green      |   60875 |  54,787,500 |
| Bright industrial |   97133 |  87,419,700 |
| Dark urban        |  515407 | 463,866,300 |


### Question 6: Compare your class areas with those of another student.

<!-- If there are systematic differences between the classifications, can the two of you explain them on the basis of the training regions that you created for each class? -->

The table below is the area in m^2 by each classification class from whom I am comparing my result with.

| Class value       |  AREA (m^2) |
| :---------------- | ----------: |
| Dark water        |  27,863,100 |
| Bright water      | 124,887,600 |
| Maroon vegetation | 619,711,200 |
| Pink field        | 468,477,000 |
| Dark green        |   2,097,700 |
| Bright green      | 119,602,800 |
| Bright industrial | 890,351,100 |
| Dark urban        | 150,399,900 |

The the largest differences between our classification occur in the `bright industrial` and `dark urban`, and `pink-classified` classes. The other student classified 890.35km^2 as bright urban/industrial, compare with my 87.42km^2. They explicitly told me they included the suburban area into the `bright industrial` in that class, whereas my classification assigned more area to `dark urban` and `pink fields`. My `pink-field` area was 925.74km^2 compared with their 468.48km^2, and my `dark urban` areas was 463.87km compared with their 150.40km^2. These differences are consistent with different choices of training regions fore mixed residential landscape containing, roads, lawns, and trees.

Due to the missing parts of both reservoir, I find it reasonable to expect a lower area coverage for dark water. In the case of bright water,

We have pretty similar coverage about 600 million of maroon vegetation and

The student I compared with also includes the suburban as bright industrial land cover, which explains the difference between our `bright industrial` class.

However, , the `pink field` class is over classified. Looking from `Baltimore_tm_subset.img`, the field that should have been classified as `dark urban`, dense residential area in the city are classified as pink field.

Along with some maroon vegetation,

Unless the contrast exceptionally noticeable, like Patterson Park in East baltimore

Although our `maroon-vegetation` areas were relatively similar, our `pink-field` areas differed substantially. My classification assigned 457.27 km^2 more to pink fields than the other student’s. This is consistent with their inclusion of suburban areas in bright urban/industrial, while my classification assigned many vegetated residential pixels to pink fields. My visual comparison also showed apparent tree canopy being classified as pink fields (as mentioned in Q2). These observations suggest that our training samples separated grass, tree cover, and mixed suburban surfaces differently.


## Unsupervised Classification


**Classification Area by Class ID**

| Class ID       | Dark water | Bright water | Maroon vegetation | Pink field | Dark green | Bright green | Bright industrial | Dark urban |
| :------------- | ---------: | -----------: | ----------------: | ---------: | ---------: | -----------: | ----------------: | ---------: |
| 0 (Grey)       |          0 |            0 |             73800 |  182429100 |    5841000 |        16200 |            909900 |  144669600 |
| 1 (White)      |          0 |       421200 |               900 |     157500 |     149400 |     13696200 |          34586100 |   27820800 |
| 2 (Red)        |          0 |            0 |         117733500 |  304529400 |    3105000 |            0 |                 0 |       1800 |
| 3 (Blue)       |   13930200 |    106890300 |            794700 |          0 |     155700 |            0 |          50262300 |    4518900 |
| 4 (Pink)       |          0 |            0 |           1018800 |  281655900 |          0 |            0 |              9900 |      44100 |
| 5 (Crimson)    |          0 |            0 |         549214200 |   76924800 |   29587500 |            0 |            793800 |    6217200 |
| 6 (Green)      |          0 |            0 |             90900 |   78345000 |   62925300 |     41075100 |            405900 |   34381800 |
| 7 (Dark green) |          0 |            0 |                 0 |    1702800 |          0 |            0 |            451800 |  246212100 |


### Question 7: Similarities & differences between your supervised and unsupervised.

<!-- Comment on the similarities and differences between your Supervised and Unsupervised Images. Is either one superior than the other for particular classes? [Note: you can dig into the details using Spatial Analyst → Zonal → Tabulate Areas, but that’s optional] -->

Overlap area (m^2) between unsupervised classes (rows) and supervised (using ISO Cluster Classifier) classes (columns).

Both classification show similar broad patterns of water, forest, and urban development, but their classes do not correspond one-to-one. The supervised classification uses predefined training classes, whereas ISO method used classifies object base on their spectral values across all input band, not just the displayed bands.

Overall, water is more consistently grouped in the unsupervised result. Class 3 contains area labeled both `dark` and `bright water`, along with the area falsely labelled as `bright industrial` in the supervised map. Visually, parts of the river and reservoirs that were incorrectly labeled industrial and included in the unsupervised water class. This suggests that the unsupervised result is better for delineating these water bodies.

The unsupervised classifier does not follow the predefined class label as the supervised classifier Therefore, it does not distinguish between `dark water` and `bright water`, all dark blue objects and those close to that colour are all classified into a single class.

and because it runs without a pre-labeled training class, it successfully categorizes all water, without including parts of the River channel as `bright industrial`spilling over into categorizing parts of river as `bright industrial` repeating the same systematic error saw earlier in the supervised classification.

The vegetation classes between the supervised and unsupervised also show substantial agreement: approximately 82.1% of the supervised maroon vegetation falls within unsupervised class 5. However, the supervised `pink field` class is distributed across several unsupervised classes, particularly classes 0, 2, and 4. Dark urban areas are also divided among several clusters. These differences indicate that the supervised classification separates spectral variation within some of the broader supervised categories.

The supervised approach provides classes with predefined land-cover meanings, while the supervised result requires interpretation of each cluster. In this scene, the unsupervised result appears better for water delineation, but independent reference data is needed to determine which method is more accurate for vegetation and urban land.

<!-- distinguish between buildings more, than with vegetation and water-body  -->


### Question 8: Include a screen capture of both supervised and unsupervised classifications.

|             Supervised Classification             |             Unsupervised Classification             |
| :-----------------------------------------------: | :-------------------------------------------------: |
| ![Supervised](figs/PHOTO-2026-10-10-02-16-55.jpg) | ![Unsupervised](figs/PHOTO-2026-10-10-02-17-39.jpg) |


## Post-processing:

### Question 9:

#### Overall accuracy

$$
\begin{gather}

    \text{Overall correct} = \frac{\text{Total correct}}{\text{ Total of pixels}} = \frac{482}{\text{584}}\\
    \text{Overall correct} = 82.5\%

\end{gather}
$$

#### Kappa Coefficient

Total number of pixel ($N$) = 584

<!-- $$
\begin{gather}
    N = (92 + 14 + 4) + (2 + 42 +3 + 1) + (1 + 8 +55 +15) + (1 + 41 +2+6+1) +  \\
    (2+11+75+9+2) + (2+1+5+4+141+3) + (1+4+36) = 584
\end{gather}
$$ -->

Diagonal sum ($A$):

$$
\begin{equation}
A =92 + 42 + 55 + 41 + 75 + 141 + 36 = 482
\end{equation}
$$

Sum of row times column ($B$):

$$
\begin{equation}
B = 96 \times 110 + 66 \times 48 + 65\times 79 + 57 \times 51 + 82 \times 99 + 176 \times 156 + 42 \times 41 = 59066
\end{equation}
$$

The kappa coefficient is given as,

$$
\begin{gather}
    K = \frac{N\times A - B}{N^2 - B} = \frac{584 \times 482 - 59066}{584^2 - 59066}\\
    K  = 0.789
\end{gather}
$$

#### User's & Producer's accuracy

<!-- ![matrix](<figs/Screenshot 2026-10-11 at 11.47.04 PM.png>)

- User: For each class: Correct pixels / # pixels placed in the class
- Producer For each class: Correct pixels / # pixels that should have been placed in the class

-->

- From the user (commission) perspective `dense urban` is causing the most problem.

- From the producer (omission) perspective `mixed industrial ` is causing the most problem.

| Class                | User accuracy     | Producer's accuracy |
| -------------------- | ----------------- | ------------------- |
| Water                | 92 / 110 = 83.6%  | 92 / 96 = 95.8%     |
| dense urban          | 42 / 48 = 87.5%   | 42 / 66 = 63.6%     |
| mixed industrial     | 55 / 79 = 69.6%   | 55 / 65 = 84.6      |
| coniferous forest    | 41 / 51 = 80.3%   | 41 / 57 = 71.9%     |
| broadleaf forest     | 75 / 99 = 75.8%   | 75 / 82 = 91.4%     |
| Suburban residential | 141 / 156 = 90.4% | 141 / 176 = 80.1%   |
| Agriculture          | 36 / 41 = 87.8%   | 36 / 42 = 85.7%     |
